In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer

movies = pd.read_csv('movies.csv')
ratings = pd.read_csv('ratings.csv')

In [39]:
# First Look ritual

movies.head()
movies.info()
movies.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB


,movieId
count,9742.000000
mean,42200.353623
std,52160.494854
min,1.000000
25%,3248.250000
50%,7300.000000
75%,76232.000000
max,193609.000000


In [40]:
movies

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy
...,...,...,...
9737,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy
9738,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy
9739,193585,Flint (2017),Drama
9740,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation


In [41]:
ratings

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [42]:
# First Look ritual

ratings.head()
ratings.info()
ratings.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


,userId,movieId,rating,timestamp
count,100836.000000,100836.000000,100836.000000,1.008360e+05
mean,326.127564,19435.295718,3.501557,1.205946e+09
std,182.618491,35530.987199,1.042529,2.162610e+08
min,1.000000,1.000000,0.500000,8.281246e+08
25%,177.000000,1199.000000,3.000000,1.019124e+09
50%,325.000000,2991.000000,3.500000,1.186087e+09
75%,477.000000,8122.000000,4.000000,1.435994e+09
max,610.000000,193609.000000,5.000000,1.537799e+09


In [43]:
# Checking Missing Values
print('----- movies.csv -----')
print(movies.isnull().sum())
print()

print('----- ratings.csv -----')
print(ratings.isnull().sum())

----- movies.csv -----
movieId    0
title      0
genres     0
dtype: int64

----- ratings.csv -----
userId       0
movieId      0
rating       0
timestamp    0
dtype: int64


In [4]:
# Merging Data Frames

merged_df = pd.merge(movies, ratings, on='movieId')
merged_df

,movieId,title,genres,userId,rating,timestamp
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1,4.0,964982703
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,5,4.0,847434962
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,7,4.5,1106635946
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,15,2.5,1510577970
4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,17,4.5,1305696483
...,...,...,...,...,...,...
100831,193581,Black Butler: Book of the Atlantic (2017),Action|Animation|Comedy|Fantasy,184,4.0,1537109082
100832,193583,No Game No Life: Zero (2017),Animation|Comedy|Fantasy,184,3.5,1537109545
100833,193585,Flint (2017),Drama,184,3.5,1537109805
100834,193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation,184,3.5,1537110021


In [3]:
# Content Recommender Engine

tfidf = TfidfVectorizer()
genre_matrix = tfidf.fit_transform(movies['genres'])
cosine_sim = cosine_similarity(genre_matrix)

def content_recommender(title, num=3):
  indx = movies[movies['title'] == title].index[0]
  scores = list(enumerate(cosine_sim[indx]))
  scores = sorted(scores, key=lambda x: x[1], reverse=True)
  top = [movies['title'][i] for i,_ in scores[1:num+1]]
  return top

print(content_recommender('Toy Story (1995)'))

['Antz (1998)', 'Toy Story 2 (1999)', 'Adventures of Rocky and Bullwinkle, The (2000)']


In [6]:
# Collaborative Filtering

user_matrix = merged_df.pivot_table(index='userId', columns='title', values='rating').fillna(0)
title_matrix = user_matrix.T
title_matrix_sim = cosine_similarity(title_matrix)
title_matrix_sim_df = pd.DataFrame(title_matrix_sim, index=title_matrix.index, columns=title_matrix.index)
title_matrix_sim_df.round(2)

title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
title,,,,,,,,,,,,,,,,,,,,,
'71 (2014),1.00,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.14,0.00,...,0.00,0.34,0.54,0.71,0.0,0.00,0.14,0.33,0.00,0.0
'Hellboy': The Seeds of Creation (2004),0.00,1.00,0.71,0.00,0.00,0.0,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.0
'Round Midnight (1986),0.00,0.71,1.00,0.00,0.00,0.0,0.18,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.0
'Salem's Lot (2004),0.00,0.00,0.00,1.00,0.86,0.0,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.0
'Til There Was You (1997),0.00,0.00,0.00,0.86,1.00,0.0,0.00,0.00,0.00,0.00,...,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
eXistenZ (1999),0.00,0.00,0.00,0.00,0.00,0.0,0.21,0.22,0.10,0.13,...,0.00,0.00,0.00,0.00,0.0,1.00,0.19,0.00,0.17,0.0
xXx (2002),0.14,0.00,0.00,0.00,0.00,0.0,0.09,0.00,0.28,0.02,...,0.07,0.31,0.17,0.25,0.0,0.19,1.00,0.27,0.10,0.0
xXx: State of the Union (2005),0.33,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.16,0.00,...,0.00,0.38,0.18,0.23,0.0,0.00,0.27,1.00,0.00,0.0


In [8]:
# Collaborative Recommender Engine

def collab_recommender(title, num=3):
  indx = title_matrix.index.get_loc(title)
  scores = list(enumerate(title_matrix_sim[indx]))
  scores = sorted(scores, key=lambda x: x[1], reverse=True)
  top = [title_matrix.index[i] for i,_ in scores[1:num+1]]
  return top

print(collab_recommender('Toy Story (1995)'))

['Toy Story 2 (1999)', 'Jurassic Park (1993)', 'Independence Day (a.k.a. ID4) (1996)']


In [9]:
# Hybrid Recommender Engine

def hybrid_recommender(title, w_content=0.5, w_collab=0.5):
  indx = movies[movies['title'] == title].index[0]
  content_scores = cosine_sim[indx]

  if title in title_matrix_sim_df.columns:
    collab_scores = title_matrix_sim_df[title].reindex(movies['title']).fillna(0).values
  else:
    collab_scores = np.zeros(len(movies))

  hybrid_scores = w_content * content_scores + w_collab * collab_scores
  hybrid_scores = list(enumerate(hybrid_scores))
  hybrid_scores = sorted(hybrid_scores, key=lambda x: x[1], reverse=True)
  top = [movies['title'][i] for i,_ in hybrid_scores[1:4]]
  return top

print(hybrid_recommender('Toy Story (1995)'))

['Toy Story 2 (1999)', 'Monsters, Inc. (2001)', 'Shrek (2001)']
